In [7]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
from sklearn.linear_model import LinearRegression
from sklearn.metrics import r2_score

# 1. Carregar os arquivos do Data Mart e o arquivo de Despesas do TSE usando o separador correto
df_votos = pd.read_csv('fato_votos.csv', sep=';')
df_cand = pd.read_csv('dim_candidato.csv', sep=';')

# Carrega apenas as colunas essenciais do arquivo gigante do TSE para economizar memória RAM no Colab
df_despesas = pd.read_csv(
    'despesas_contratadas_candidatos_2020_SP.csv',
    sep=';',
    encoding='latin1',
    usecols=['SQ_CANDIDATO', 'VR_DESPESA_CONTRATADA']
)

# 2. Tratar e agrupar os gastos por candidato
# Converte o texto com vírgula (ex: "1500,50") para número decimal (1500.50)
df_despesas['VR_DESPESA_CONTRATADA'] = (
    df_despesas['VR_DESPESA_CONTRATADA']
    .astype(str)
    .str.replace(',', '.')
    .astype(float)
)

gastos_por_cand = (
    df_despesas.groupby('SQ_CANDIDATO')['VR_DESPESA_CONTRATADA']
    .sum()
    .reset_index()
    .rename(columns={'SQ_CANDIDATO': 'sq_candidato', 'VR_DESPESA_CONTRATADA': 'vr_despesa'})
)

# 3. Agrupar os votos por candidato (usando dim_candidato como ponte para pegar o sq_candidato)
votos_por_id = df_votos.groupby('id_candidato')['qt_votos'].sum().reset_index()
df_cand_votos = pd.merge(df_cand[['id_candidato', 'sq_candidato', 'nm_urna_candidato']], votos_por_id, on='id_candidato', how='inner')

# 4. Unir Votos + Despesas na tabela final para a Regressão
df_regressao = pd.merge(df_cand_votos, gastos_por_cand, on='sq_candidato', how='inner')

print(f"Total de candidatos cruzados com sucesso: {len(df_regressao)}")
print(df_regressao.head())

Total de candidatos cruzados com sucesso: 128
   id_candidato  sq_candidato     nm_urna_candidato  qt_votos  vr_despesa
0           167  250000640972  SOCORRO DA HABITAÇÃO         1    14907.00
1           170  250000640979    RODRIGO S GONZALES         1    10000.00
2           172  250000640997            GIZA LEMOS         1    29830.10
3           175  250000641005             LÉO LEMOS         1    10000.00
4           176  250000641034             ANTONINHA         1     9964.35
